# Library imports

In [19]:
import os
import time
from typing import List, Tuple

import torch
import numpy as np
import cv2
import onnxruntime as ort
from qai_appbuilder import QNNContext, PerfProfile, Runtime, LogLevel, ProfilingLevel, QNNConfig
from qai_hub_models.models.face_det_lite.utils import detect as _face_det_decode
from sklearn.metrics import mean_squared_error, accuracy_score
import tqdm

# Dataset load

In [20]:
DATA_DIR = "/home/ubuntu/AgeDB"
img_paths = [os.path.join(DATA_DIR, img) for img in os.listdir(DATA_DIR) if img.endswith('.jpg')][:500]
age_labels = [int(img.split('_')[-2]) for img in img_paths]
gender_labels = [0 if img.split('_')[-1].split('.')[0] == 'f' else 1 for img in img_paths]

In [21]:
len(img_paths), len(age_labels), len(gender_labels)

(500, 500, 500)

In [22]:
def preprocess(img, target_size=(224, 224)):
    img = cv2.resize(img, target_size)
    img = img.astype(np.float32) / 255.0
    # img = np.transpose(img, (2, 0, 1))  # HWC to CHW
    return img[np.newaxis, :]  # Add batch dimension

In [23]:
def postprocess_age(output) -> int:
    indices = np.arange(0, 101)
    return int(np.sum(output[0] * indices))


def postprocess_gender(output) -> int:
    return int(np.argmax(output[0]))

In [24]:
FACE_DET_MODEL_NAME = "face_det_lite"
FACE_DET_MODEL_PATH = f"models/{FACE_DET_MODEL_NAME}.bin"


FACE_DET_INPUT_H = 480
FACE_DET_INPUT_W = 640
FACE_DET_STRIDE = 8
FACE_DET_FMAP_H = FACE_DET_INPUT_H // FACE_DET_STRIDE   # 60
FACE_DET_FMAP_W = FACE_DET_INPUT_W // FACE_DET_STRIDE   # 80
FACE_DET_SCORE_THRESHOLD = 0.55
FACE_DET_NMS_IOU = 0.3

class FaceDetQNN(QNNContext):
    def Inference(self, input_data):
        return super().Inference([input_data])
    
def bursted_inference(qnn, nhwc: np.ndarray):
    PerfProfile.SetPerfProfileGlobal(PerfProfile.BURST)
    try:
        return qnn.Inference(nhwc)
    finally:
        PerfProfile.RelPerfProfileGlobal()

In [25]:
def run_face_detect(
    image_bgr: np.ndarray, qnn: FaceDetQNN
) -> List[Tuple[List[float], float, List[Tuple[float, float]]]]:
    """Detect faces on a full BGR frame.

    Returns a list of (xyxy, score, landmark5) in original-image coordinates.
    """
    t0 = time.perf_counter()
    orig_h, orig_w = image_bgr.shape[:2]

    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(
        gray, (FACE_DET_INPUT_W, FACE_DET_INPUT_H), interpolation=cv2.INTER_LINEAR
    )
    nhwc = np.ascontiguousarray(
        (resized.astype(np.float32) / 255.0)[None, ..., None]
    )
    out = bursted_inference(qnn, nhwc)

    # captured_events = qnn_log_capture.drain()  # disabled
    captured_events: list = []

    hm = torch.from_numpy(
        out[0].reshape(1, FACE_DET_FMAP_H, FACE_DET_FMAP_W, 1)
    ).permute(0, 3, 1, 2)
    bx = torch.from_numpy(
        out[1].reshape(1, FACE_DET_FMAP_H, FACE_DET_FMAP_W, 4)
    ).permute(0, 3, 1, 2)
    lm = torch.from_numpy(
        out[2].reshape(1, FACE_DET_FMAP_H, FACE_DET_FMAP_W, 10)
    ).permute(0, 3, 1, 2)

    dets = _face_det_decode(
        hm, bx, lm,
        threshold=FACE_DET_SCORE_THRESHOLD,
        nms_iou=FACE_DET_NMS_IOU,
        stride=FACE_DET_STRIDE,
    )

    sx = orig_w / float(FACE_DET_INPUT_W)
    sy = orig_h / float(FACE_DET_INPUT_H)

    results = []
    for d in dets:
        x1 = float(d.x) * sx
        y1 = float(d.y) * sy
        x2 = float(d.r) * sx
        y2 = float(d.b) * sy
        x1 = max(0.0, min(x1, orig_w - 1))
        y1 = max(0.0, min(y1, orig_h - 1))
        x2 = max(0.0, min(x2, orig_w - 1))
        y2 = max(0.0, min(y2, orig_h - 1))
        if x2 <= x1 or y2 <= y1:
            continue
        landmark = []
        if d.landmark is not None:
            landmark = [(float(lx) * sx, float(ly) * sy) for lx, ly in d.landmark]
        results.append(([x1, y1, x2, y2], float(d.score), landmark))

    return results

In [26]:
def preprocess_face_crop(image_bgr: np.ndarray, xyxy: List[float],
                         landmark: List[Tuple[float, float]] | None = None,
                         target_size=(224, 224), margin: float = 0.3):
    """Align (eyes level) + margin-expand + letterbox crop. BGR, [0,1], (1,H,W,3).

    landmark: 5 pts from run_face_detect -> [left_eye, right_eye, nose, l_mouth, r_mouth].
    margin:   fraction of bbox added each side (loosen crop; aging cues live in context).
    """
    h, w = image_bgr.shape[:2]

    # 1) align: rotate whole frame about eye-center so the eye line is horizontal
    if landmark is not None and len(landmark) >= 2:
        (lx, ly), (rx, ry) = landmark[0], landmark[1]
        eye_c = ((lx + rx) / 2.0, (ly + ry) / 2.0)
        angle = float(np.degrees(np.arctan2(ry - ly, rx - lx)))
        M = cv2.getRotationMatrix2D(eye_c, angle, 1.0)
        image_bgr = cv2.warpAffine(image_bgr, M, (w, h), flags=cv2.INTER_LINEAR)

    # 2) expand bbox by margin, clamp to frame
    x1, y1, x2, y2 = xyxy
    bw, bh = x2 - x1, y2 - y1
    x1 -= bw * margin; x2 += bw * margin
    y1 -= bh * margin; y2 += bh * margin
    x1 = int(max(0, min(x1, w - 1))); y1 = int(max(0, min(y1, h - 1)))
    x2 = int(max(0, min(x2, w - 1))); y2 = int(max(0, min(y2, h - 1)))
    if x2 <= x1 or y2 <= y1:
        return None
    crop = image_bgr[y1:y2, x1:x2]   # models require BGR -> keep BGR

    # 3) letterbox to target (aspect-preserving + zero-pad), matches DeepFace resize_image
    ch, cw = crop.shape[:2]
    factor = min(target_size[0] / ch, target_size[1] / cw)
    dsize = (max(1, int(cw * factor)), max(1, int(ch * factor)))
    img = cv2.resize(crop, dsize)
    d0 = target_size[0] - img.shape[0]
    d1 = target_size[1] - img.shape[1]
    img = np.pad(
        img,
        ((d0 // 2, d0 - d0 // 2), (d1 // 2, d1 - d1 // 2), (0, 0)),
        "constant",
    )
    if img.shape[:2] != tuple(target_size):
        img = cv2.resize(img, target_size)
    return np.ascontiguousarray((img.astype(np.float32) / 255.0)[None, ...])

In [27]:
QNN_DIR = "qai_libs/"
QNNConfig.Config(str(QNN_DIR), Runtime.HTP, LogLevel.ERROR, ProfilingLevel.OFF)
face_det_qnn = FaceDetQNN(FACE_DET_MODEL_NAME, str(FACE_DET_MODEL_PATH))

/prj/qct/webtech_scratch20/mlg_user_admin/qaisw_source_repo/rel/qairt-2.45.0/release/SNPE_SRC/avante-tools/prebuilt/dsp/hexagon-sdk-5.5.5/ipc/fastrpc/rpcmem/src/rpcmem_android.c:38:dummy call to rpcmem_init, rpcmem APIs will be used from libxdsprpc


# Age Predict

In [13]:
MARGIN_RATE=0.

## Onnx

In [10]:
age_session = ort.InferenceSession("models/age.onnx")

input_name = age_session.get_inputs()[0].name
output_name = age_session.get_outputs()[0].name

In [11]:
# Warm up
dummy = np.random.rand(224, 224, 3).astype(np.float32)
test = age_session.run([output_name], {input_name: preprocess(dummy)})

In [1]:
np.sum(test[0]) # Already softmax

NameError: name 'np' is not defined

In [14]:
correct_age_labels = []
age_predicts = []
infer_times = []  # pure inference latency, per sample (ms)
kept_paths = []
t_wall_start = time.perf_counter()
for path, age_label in tqdm.tqdm(zip(img_paths, age_labels)):
    img = cv2.imread(path)
    face_det = run_face_detect(img, face_det_qnn)
    if face_det is None or len(face_det) == 0:
        print(f"Invalid number of faces in {path}, skipping.")
        continue
    xyxy, score, landmark = face_det[0]   # first detected face: (bbox, score, landmark5)
    input_data = preprocess_face_crop(img, xyxy, landmark, margin=MARGIN_RATE)

    t0 = time.perf_counter()
    output = age_session.run([output_name], {input_name: input_data})
    t1 = time.perf_counter()

    infer_times.append((t1 - t0) * 1000.0)
    age_predicts.append(postprocess_age(output))
    correct_age_labels.append(age_label)
    kept_paths.append(path)
t_wall_end = time.perf_counter()

infer_times = np.array(infer_times)
n = len(infer_times)
wall_s = t_wall_end - t_wall_start

9it [00:03,  2.41it/s]

Invalid number of faces in /home/ubuntu/AgeDB/758_WEBDuBois_30_m.jpg, skipping.


123it [00:53,  2.68it/s]

Invalid number of faces in /home/ubuntu/AgeDB/8261_ReginaldOwen_75_m.jpg, skipping.


204it [01:25,  2.59it/s]

Invalid number of faces in /home/ubuntu/AgeDB/37_EsteeLauder_65_f.jpg, skipping.


500it [03:42,  2.25it/s]


In [15]:
# Inference-only latency (industry standard: time inference call, report percentiles)
print(f"samples              : {n}")
print(f"latency mean   (ms)  : {infer_times.mean():.3f}")
print(f"latency min/max(ms)  : {infer_times.min():.3f} / {infer_times.max():.3f}")

# Throughput
single_stream_fps = 1000.0 / infer_times.mean()   # latency-bound (inference only)
e2e_fps = n / wall_s                               # end-to-end incl. imread+preprocess+post
print(f"single-stream FPS    : {single_stream_fps:.2f}  (1000 / mean infer latency)")
print(f"Pipeline FPS    : {e2e_fps:.2f}  (decode+preprocess+infer+post over wall time)")

preds = np.array(age_predicts, dtype=np.float64)
labels = np.array(correct_age_labels, dtype=np.float64)
err = preds - labels
abs_err = np.abs(err)

print(f"Process samples      : {len(preds)}")
print(f"MAE               : {abs_err.mean():.3f}")
print(f"RMSE              : {np.sqrt((err**2).mean()):.3f}")

samples              : 497
latency mean   (ms)  : 430.923
latency min/max(ms)  : 314.412 / 934.740
single-stream FPS    : 2.32  (1000 / mean infer latency)
Pipeline FPS    : 2.24  (decode+preprocess+infer+post over wall time)
Process samples      : 497
MAE               : 10.706
RMSE              : 14.145


In [ ]:
# ---- Age outlier analysis ----
# Goal: find which samples blow up the MSE, and check predictions stay in [0, 100].
preds = np.array(age_predicts, dtype=np.float64)
labels = np.array(correct_age_labels, dtype=np.float64)
paths = np.array(kept_paths)
err = preds - labels
abs_err = np.abs(err)

print(f"kept samples      : {len(preds)}")
print(f"MAE               : {abs_err.mean():.3f}")
print(f"RMSE              : {np.sqrt((err**2).mean()):.3f}")
print(f"abs err p50/p90/p99: {np.percentile(abs_err,50):.1f} / "
      f"{np.percentile(abs_err,90):.1f} / {np.percentile(abs_err,99):.1f}")

# 1) Sanity: postprocess_age = sum(prob * 0..100) -> must be in [0,100] if output is softmax.
#    Out-of-range pred = model output not normalized (bug), not a hard sample.
oob = np.where((preds < 0) | (preds > 100))[0]
print(f"\npredictions outside [0,100]: {len(oob)}")
for i in oob[:10]:
    print(f"  pred={preds[i]:.2f}  label={labels[i]:.0f}  {os.path.basename(paths[i])}")

# 2) Label sanity: AgeDB ages are 1..101-ish. Flag implausible parsed labels.
bad_label = np.where((labels < 1) | (labels > 110))[0]
print(f"\nimplausible labels (<1 or >110): {len(bad_label)}")
for i in bad_label[:10]:
    print(f"  label={labels[i]:.0f}  {os.path.basename(paths[i])}")

# 3) Worst absolute-error samples — these dominate MSE (squared).
TOPK = 25
order = np.argsort(abs_err)[::-1][:TOPK]
print(f"\nTop {TOPK} outliers by |error| (idx in kept arrays):")
print(f"{'idx':>5} {'pred':>6} {'label':>6} {'err':>7}   file")
for i in order:
    print(f"{i:>5} {preds[i]:>6.1f} {labels[i]:>6.0f} {err[i]:>+7.1f}   {os.path.basename(paths[i])}")

# 4) How much does the tail cost? MSE with worst-K removed.
keep = np.argsort(abs_err)[:-TOPK]
mse_all = (err**2).mean()
mse_trim = (err[keep]**2).mean()
print(f"\nMSE all={mse_all:.2f}  MSE without top{TOPK}={mse_trim:.2f}  "
      f"(tail accounts for {100*(mse_all-mse_trim)/mse_all:.1f}% of MSE)")

kept samples      : 497
MAE               : 8.429
RMSE              : 11.133
abs err p50/p90/p99: 7.0 / 18.0 / 32.2

predictions outside [0,100]: 0

implausible labels (<1 or >110): 0

Top 25 outliers by |error| (idx in kept arrays):
  idx   pred  label     err   file
   29   41.0     86   -45.0   13657_JuneAllyson_86_f.jpg
  224   35.0     78   -43.0   11641_MaureenOHara_78_f.jpg
  117   54.0     93   -39.0   2358_AbeVigoda_93_m.jpg
  464   54.0     92   -38.0   5176_BobHope_92_m.jpg
    3   43.0     80   -37.0   15697_KathrynGrayson_80_f.jpg
  406   42.0     74   -32.0   12898_RuthGordon_74_f.jpg
  367   57.0     87   -30.0   464_ConradHilton_87_m.jpg
  334   44.0     74   -30.0   12083_ElsaMartinelli_74_f.jpg
    7   42.0     70   -28.0   4153_GenaRowlands_70_m.jpg
  124   44.0     17   +27.0   1444_FriedrichNietzsche_17_m.jpg
  112   44.0     70   -26.0   16405_Ross_70_f.jpg
  304   42.0     68   -26.0   501_motherTereza_68_f.jpg
  446   66.0     92   -26.0   1229_ClaudeL╨ТviStraus

## QNN

In [34]:
class AgeDetectQNN(QNNContext):
    def Inference(self, input_data):
        return super().Inference([input_data])

age_qnn = AgeDetectQNN("age", "models/age.bin")

In [35]:
# Warm up
dummy = np.random.rand(224, 224, 3).astype(np.float32)
age_qnn.Inference(preprocess(dummy))

[array([[1.69873310e-05, 8.30841158e-03, 5.77163743e-03, 3.47137474e-03,
         1.02310190e-02, 7.61032151e-03, 7.70950364e-03, 4.27246140e-03,
         3.26156639e-03, 3.02696251e-03, 4.68826341e-03, 4.96292161e-03,
         6.74819993e-03, 7.40432786e-03, 7.15637254e-03, 7.20977830e-03,
         7.28225755e-03, 8.97979829e-03, 8.41522310e-03, 9.67407320e-03,
         1.22070322e-02, 1.32370004e-02, 1.39312753e-02, 1.56402607e-02,
         1.57775898e-02, 1.86767597e-02, 1.72882099e-02, 1.91192646e-02,
         1.57318134e-02, 1.84936542e-02, 1.68151874e-02, 1.89056415e-02,
         1.60064716e-02, 1.69067401e-02, 1.64032001e-02, 1.53732309e-02,
         1.63574237e-02, 1.42288217e-02, 1.50222788e-02, 1.43432627e-02,
         1.20849619e-02, 1.28173837e-02, 1.25045786e-02, 1.42745981e-02,
         1.36032114e-02, 1.30920419e-02, 1.37557993e-02, 1.43508920e-02,
         1.36032114e-02, 1.48010263e-02, 1.39999399e-02, 1.44729624e-02,
         1.54647836e-02, 1.48391733e-02, 1.19323740

In [36]:
correct_age_labels = []
age_predicts = []
infer_times = []  # pure inference latency, per sample (ms)
kept_paths = []
t_wall_start = time.perf_counter()
for path, age_label in tqdm.tqdm(zip(img_paths, age_labels)):
    img = cv2.imread(path)
    face_det = run_face_detect(img, face_det_qnn)
    if face_det is None or len(face_det) == 0:
        print(f"Invalid number of faces in {path}, skipping.")
        continue
    
    xyxy, score, landmark = face_det[0]   # first detected face: (bbox, score, landmark5)
    input_data = preprocess_face_crop(img, xyxy, landmark, margin=MARGIN_RATE)

    t0 = time.perf_counter()
    output = bursted_inference(age_qnn, input_data)
    t1 = time.perf_counter()

    infer_times.append((t1 - t0) * 1000.0)
    age_predicts.append(postprocess_age(output))
    correct_age_labels.append(age_label)
    kept_paths.append(path)
t_wall_end = time.perf_counter()

infer_times = np.array(infer_times)
n = len(infer_times)
wall_s = t_wall_end - t_wall_start

15it [00:00, 45.35it/s]

Invalid number of faces in /home/ubuntu/AgeDB/758_WEBDuBois_30_m.jpg, skipping.


132it [00:03, 42.42it/s]

Invalid number of faces in /home/ubuntu/AgeDB/8261_ReginaldOwen_75_m.jpg, skipping.


212it [00:05, 41.70it/s]

Invalid number of faces in /home/ubuntu/AgeDB/37_EsteeLauder_65_f.jpg, skipping.


500it [00:12, 40.10it/s]


In [37]:
# Inference-only latency (industry standard: time inference call, report percentiles)
print(f"samples              : {n}")
print(f"latency mean   (ms)  : {infer_times.mean():.3f}")
print(f"latency min/max(ms)  : {infer_times.min():.3f} / {infer_times.max():.3f}")

# Throughput
single_stream_fps = 1000.0 / infer_times.mean()   # latency-bound (inference only)
e2e_fps = n / wall_s                               # end-to-end incl. imread+preprocess+post
print(f"single-stream FPS    : {single_stream_fps:.2f}  (1000 / mean infer latency)")
print(f"Pipeline FPS    : {e2e_fps:.2f}  (decode+preprocess+infer+post over wall time)")

preds = np.array(age_predicts, dtype=np.float64)
labels = np.array(correct_age_labels, dtype=np.float64)
err = preds - labels
abs_err = np.abs(err)

print(f"Process samples      : {len(preds)}")
print(f"MAE               : {abs_err.mean():.3f}")
print(f"RMSE              : {np.sqrt((err**2).mean()):.3f}")

samples              : 497
latency mean   (ms)  : 13.644
latency min/max(ms)  : 11.986 / 19.355
single-stream FPS    : 73.29  (1000 / mean infer latency)
Pipeline FPS    : 39.84  (decode+preprocess+infer+post over wall time)
Process samples      : 497
MAE               : 8.505
RMSE              : 11.209


In [38]:
if face_det is not None:
    del face_det
    face_det = None

if age_qnn is not None:
    del age_qnn
    age_qnn = None

# Gender Predict

In [28]:
MARGIN_RATE = 0.

## ONNX

In [29]:
gender_session = ort.InferenceSession("models/gender.onnx")
input_name = gender_session.get_inputs()[0].name
output_name = gender_session.get_outputs()[0].name

In [30]:
# Warm up
dummy = np.random.rand(224, 224, 3).astype(np.float32)
test = gender_session.run([output_name], {input_name: preprocess(dummy)})

In [31]:
np.sum(test[0]) 

np.float32(1.0)

In [32]:
test[0]

array([[0.49551222, 0.50448775]], dtype=float32)

In [33]:
correct_gender_labels = []
gender_predicts = []
infer_times = []  # pure inference latency, per sample (ms)
kept_paths = []
t_wall_start = time.perf_counter()
for path, gender_label in tqdm.tqdm(zip(img_paths, gender_labels)):
    img = cv2.imread(path)
    face_det = run_face_detect(img, face_det_qnn)
    if face_det is None or len(face_det) == 0:
        print(f"Invalid number of faces in {path}, skipping.")
        continue
    xyxy, score, landmark = face_det[0]   # first detected face: (bbox, score, landmark5)
    input_data = preprocess_face_crop(img, xyxy, landmark, margin=MARGIN_RATE)

    t0 = time.perf_counter()
    output = gender_session.run([output_name], {input_name: input_data})
    t1 = time.perf_counter()

    infer_times.append((t1 - t0) * 1000.0)
    gender_predicts.append(postprocess_gender(output))
    correct_gender_labels.append(gender_label)
    kept_paths.append(path)
t_wall_end = time.perf_counter()

infer_times = np.array(infer_times)
n = len(infer_times)
wall_s = t_wall_end - t_wall_start

9it [00:04,  2.19it/s]

Invalid number of faces in /home/ubuntu/AgeDB/758_WEBDuBois_30_m.jpg, skipping.


123it [00:54,  2.23it/s]

Invalid number of faces in /home/ubuntu/AgeDB/8261_ReginaldOwen_75_m.jpg, skipping.


204it [01:30,  2.08it/s]

Invalid number of faces in /home/ubuntu/AgeDB/37_EsteeLauder_65_f.jpg, skipping.


500it [03:48,  2.19it/s]


In [34]:
# Inference-only latency (industry standard: time inference call, report percentiles)
print(f"samples              : {n}")
print(f"latency mean   (ms)  : {infer_times.mean():.3f}")
print(f"latency min/max(ms)  : {infer_times.min():.3f} / {infer_times.max():.3f}")

# Throughput
single_stream_fps = 1000.0 / infer_times.mean()   # latency-bound (inference only)
e2e_fps = n / wall_s                               # end-to-end incl. imread+preprocess+post
print(f"single-stream FPS    : {single_stream_fps:.2f}  (1000 / mean infer latency)")
print(f"Pipeline FPS    : {e2e_fps:.2f}  (decode+preprocess+infer+post over wall time)")

preds = np.array(gender_predicts, dtype=np.float64)
labels = np.array(correct_gender_labels, dtype=np.float64)
acc = accuracy_score(labels, preds)
print(f"Accuracy           : {acc:.4f}")

samples              : 497
latency mean   (ms)  : 445.134
latency min/max(ms)  : 363.932 / 658.376
single-stream FPS    : 2.25  (1000 / mean infer latency)
Pipeline FPS    : 2.18  (decode+preprocess+infer+post over wall time)
Accuracy           : 0.9577


## QNN

In [35]:
class GenderDetectQNN(QNNContext):
    def Inference(self, input_data):
        return super().Inference([input_data])

gender_qnn = GenderDetectQNN("gender", "models/gender.bin")

In [36]:
# Warm up
dummy = np.random.rand(224, 224, 3).astype(np.float32)
gender_qnn.Inference(preprocess(dummy))

[array([[0.49462894, 0.503418  ]], dtype=float32)]

In [37]:
correct_gender_labels = []
gender_predicts = []
infer_times = []  # pure inference latency, per sample (ms)
kept_paths = []
t_wall_start = time.perf_counter()
for path, gender_label in tqdm.tqdm(zip(img_paths, gender_labels)):
    img = cv2.imread(path)
    face_det = run_face_detect(img, face_det_qnn)
    if face_det is None or len(face_det) == 0:
        print(f"Invalid number of faces in {path}, skipping.")
        continue
    
    xyxy, score, landmark = face_det[0]   # first detected face: (bbox, score, landmark5)
    input_data = preprocess_face_crop(img, xyxy, landmark, margin=MARGIN_RATE)

    t0 = time.perf_counter()
    output = bursted_inference(gender_qnn, input_data)
    t1 = time.perf_counter()

    infer_times.append((t1 - t0) * 1000.0)
    gender_predicts.append(postprocess_gender(output))
    correct_gender_labels.append(gender_label)
    kept_paths.append(path)
t_wall_end = time.perf_counter()

infer_times = np.array(infer_times)
n = len(infer_times)
wall_s = t_wall_end - t_wall_start

14it [00:00, 43.47it/s]

Invalid number of faces in /home/ubuntu/AgeDB/758_WEBDuBois_30_m.jpg, skipping.


131it [00:03, 42.00it/s]

Invalid number of faces in /home/ubuntu/AgeDB/8261_ReginaldOwen_75_m.jpg, skipping.


211it [00:05, 42.42it/s]

Invalid number of faces in /home/ubuntu/AgeDB/37_EsteeLauder_65_f.jpg, skipping.


500it [00:12, 40.41it/s]


In [38]:
# Inference-only latency (industry standard: time inference call, report percentiles)
print(f"samples              : {n}")
print(f"latency mean   (ms)  : {infer_times.mean():.3f}")
print(f"latency min/max(ms)  : {infer_times.min():.3f} / {infer_times.max():.3f}")

# Throughput
single_stream_fps = 1000.0 / infer_times.mean()   # latency-bound (inference only)
e2e_fps = n / wall_s                               # end-to-end incl. imread+preprocess+post
print(f"single-stream FPS    : {single_stream_fps:.2f}  (1000 / mean infer latency)")
print(f"Pipeline FPS    : {e2e_fps:.2f}  (decode+preprocess+infer+post over wall time)")

preds = np.array(gender_predicts, dtype=np.float64)
labels = np.array(correct_gender_labels, dtype=np.float64)
acc = accuracy_score(labels, preds)
print(f"Accuracy           : {acc:.4f}")

samples              : 497
latency mean   (ms)  : 13.248
latency min/max(ms)  : 12.165 / 16.422
single-stream FPS    : 75.48  (1000 / mean infer latency)
Pipeline FPS    : 40.15  (decode+preprocess+infer+post over wall time)
Accuracy           : 0.9577


In [40]:
if face_det is not None:
    del face_det
    face_det = None

# if age_qnn is not None:
#     del age_qnn
#     age_qnn = None

if gender_qnn is not None:
    del gender_qnn
    gender_qnn = None

# Age calibration fit

IMDB-WIKI age model mean-reverts toward ~25-35: elderly under-predicted,
young (esp. young female) over-predicted. The error is **proportional** (grows
with distance from the center), so a flat `+5` offset barely helps. Fit a
per-gender affine map `true ≈ a*pred + b` by least squares on AgeDB.

Equivalent pivot/gain stretch: `gain = a`, `pivot = b / (1 - a)`.
Paste the printed coeffs into `app/utils.py` `calibrate_age`.

In [ ]:
# Collect raw age predictions + true age/gender on AgeDB (onnx age model).
# Uses face_det_qnn + age_session (both still loaded above).
MARGIN_RATE = 0.

age_session = ort.InferenceSession("models/age.onnx")
age_in = age_session.get_inputs()[0].name
age_out = age_session.get_outputs()[0].name

fit_pred, fit_true_age, fit_true_gender = [], [], []
for path, age_label, gender_label in tqdm.tqdm(
    zip(img_paths, age_labels, gender_labels), total=len(img_paths)
):
    img = cv2.imread(path)
    det = run_face_detect(img, face_det_qnn)
    if not det:
        continue
    xyxy, score, landmark = det[0]
    inp = preprocess_face_crop(img, xyxy, landmark, margin=MARGIN_RATE)
    out = age_session.run([age_out], {age_in: inp})
    fit_pred.append(postprocess_age(out))
    fit_true_age.append(age_label)
    fit_true_gender.append(gender_label)  # 0=Female, 1=Male

fit_pred = np.array(fit_pred, dtype=np.float64)
fit_true_age = np.array(fit_true_age, dtype=np.float64)
fit_true_gender = np.array(fit_true_gender, dtype=np.int64)
print("collected", len(fit_pred), "samples")

In [ ]:
# Fit per-gender affine calibration  true ~ a*pred + b  (least squares).
def fit_affine(p, t):
    A = np.vstack([p, np.ones_like(p)]).T
    a, b = np.linalg.lstsq(A, t, rcond=None)[0]
    return float(a), float(b)

def mae(p, t):
    return float(np.mean(np.abs(p - t)))

def report(tag, p, t):
    a, b = fit_affine(p, t)
    cal = a * p + t * 0 + b
    pivot = b / (1 - a) if abs(1 - a) > 1e-6 else float("nan")
    print(f"{tag:8} n={len(p):3d}  a={a:.4f} b={b:+.4f}  "
          f"(gain={a:.3f} pivot={pivot:.1f})  "
          f"MAE {mae(p, t):.2f} -> {mae(cal, t):.2f}")
    return a, b

print("                                   raw -> calibrated")
report("ALL", fit_pred, fit_true_age)

AGE_CALIB = {}
for g, name in [(0, "Female"), (1, "Male")]:
    m = fit_true_gender == g
    if m.sum() < 2:
        continue
    a, b = report(name, fit_pred[m], fit_true_age[m])
    AGE_CALIB[name] = (round(a, 4), round(b, 4))

print("\n# paste into app/utils.py")
print("AGE_CALIB =", AGE_CALIB)

# Sanity: per-decile residual (mean err = pred - true) to confirm the slope shape.
print("\npred-age decile | mean_pred mean_true mean_err  n")
edges = np.percentile(fit_pred, np.arange(0, 101, 10))
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (fit_pred >= lo) & (fit_pred <= hi)
    if m.sum() == 0:
        continue
    print(f"  [{lo:5.1f},{hi:5.1f}] | {fit_pred[m].mean():8.1f} "
          f"{fit_true_age[m].mean():8.1f} {(fit_pred[m]-fit_true_age[m]).mean():+8.1f}  {m.sum():3d}")

# Gender threshold debias

IMDB-WIKI is male-skewed, so the model's boundary leans Male (real-world: girls
called "Male"). Instead of argmax, predict Female when `p(Female) >= τ`. Sweep τ
to maximize **balanced accuracy** (mean of per-class recall) — that's what
fixes the minority (Female) class, unlike plain accuracy which a male-skewed set
rewards for guessing Male. Paste the best τ into `GENDER_FEMALE_THRESHOLD`.

In [ ]:
# Collect p(Female) + true gender, sweep the Female threshold.
gender_session = ort.InferenceSession("models/gender.onnx")
g_in = gender_session.get_inputs()[0].name
g_out = gender_session.get_outputs()[0].name

p_female, g_true = [], []
for path, gender_label in tqdm.tqdm(zip(img_paths, gender_labels), total=len(img_paths)):
    img = cv2.imread(path)
    det = run_face_detect(img, face_det_qnn)
    if not det:
        continue
    xyxy, score, landmark = det[0]
    inp = preprocess_face_crop(img, xyxy, landmark, margin=0.)
    out = gender_session.run([g_out], {g_in: inp})[0]
    p_female.append(float(out[0][0]))   # index 0 = Female
    g_true.append(gender_label)         # 0=Female, 1=Male

p_female = np.array(p_female)
g_true = np.array(g_true)               # 0=F, 1=M
is_f = g_true == 0
print(f"n={len(g_true)}  Female={is_f.sum()}  Male={(~is_f).sum()}  "
      f"(prior skew Female={is_f.mean():.2%})")

print("\n  tau | acc  bal_acc  F_recall  M_recall  F_called")
best = (-1, 0.5)
for tau in np.arange(0.20, 0.81, 0.05):
    pred_f = p_female >= tau                 # True -> Female
    acc = np.mean(pred_f == is_f)
    f_rec = np.mean(pred_f[is_f]) if is_f.any() else 0.0          # recall Female
    m_rec = np.mean(~pred_f[~is_f]) if (~is_f).any() else 0.0     # recall Male
    bal = 0.5 * (f_rec + m_rec)
    print(f"  {tau:.2f}| {acc:.3f}  {bal:.3f}    {f_rec:.3f}     {m_rec:.3f}     {pred_f.mean():.2%}")
    if bal > best[0]:
        best = (bal, round(float(tau), 2))

print(f"\nbest balanced-acc tau = {best[1]}  (bal_acc={best[0]:.3f})")
print("# paste into app/inference.py / docker-compose env")
print(f"GENDER_FEMALE_THRESHOLD = {best[1]}")